\*Please Run in colab

# Setup

### Environment Setup

In [2]:
!pip install instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 158.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 161.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 44.3 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.12.0
    Uninstalling jiter-0.12.0:
      Successfully uninstalled jiter-0.12.0


In [3]:
!cd /content
!rm -rf macro_financial_forecasting

In [4]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1983, done.
remote: Counting objects: 100% (690/690), done.
remote: Compressing objects: 100% (246/246), done.
remote: Total 1983 (delta 528), reused 454 (delta 444), pack-reused 1293 (from 2)
Receiving objects: 100% (1983/1983), 39.64 MiB | 14.51 MiB/s, done.
Resolving deltas: 100% (1271/1271), done.


In [5]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Mount GDrive

In [6]:
# from google.colab import drive

# # This will prompt you to authorize Colab to access your Google Drive.
# drive.mount('/content/gdrive')
# GDRIVE_PATH = "/content/gdrive/MyDrive/macro_financial_forecasting_files/"

### Code Setup

In [7]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")
print(f"Config: {config}")

Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'https://feeds.bloomberg.com/real-e

# Agent

In [8]:
!pip install feedparser

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.8 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=bbe0ba549850383a4bb61194c422e21aca7d61a320bae1f890f737f1f367e476
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [9]:
!pip install langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 7.1 MB/s eta 0:00:00


In [28]:
from typing import List, Dict, Any
from datetime import datetime, timedelta, timezone
import feedparser
import pandas as pd

from langchain.tools import tool
# from langchain_openai import ChatOpenAI
# from langgraph.prebuilt import create_react_agent


In [24]:

# def get_bloomberg_rss_feeds(days: int = 1) -> List[Dict[str, str]]:
#     """Fetch Bloomberg RSS news items for the last N days."""

#     from config import Config
#     from data_model.bloomberg_news_entry import BloombergNewsEntry
#     config = Config()
#     feeds = config.rss_feeds[:2]

#     cutoff = datetime.now(timezone.utc) - timedelta(days=days)
#     news = []

#     for url in feeds:
#         feed = feedparser.parse(url)
#         for entry in feed.entries:
#             if hasattr(entry, "published_parsed"):
#                 published = datetime(*entry.published_parsed[:6], tzinfo=timezone.utc)
#                 if published > cutoff:
#                     news.append({
#                         "Headline": entry.title,
#                         "Link": entry.link,
#                         "Article": entry.summary,
#                         "Date": published.isoformat(),
#                     })
#     return news

In [31]:
# mock_data = get_bloomberg_rss_feeds()
# pd.DataFrame(mock_data).to_csv("/content/mock_data.csv", index=False)

In [42]:
@tool
def get_mock_bloomberg_rss_feeds() -> List[Dict[str, str]]:
  """Fetch mock news"""
  data = [{'Headline': 'Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap',
  'Article': 'Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.',
  'Date': '2025-11-27T22:29:53+00:00'},
 {'Headline': 'Gold Poised for Fourth Monthly Gain on Fed Rate-Cut Optimism',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/gold-poised-for-fourth-monthly-gain-on-fed-rate-cut-optimism',
  'Article': 'Gold was steady &mdash; and on track for a fourth monthly gain &mdash; as signs the Federal Reserve will cut rates next month supported the precious metal.',
  'Date': '2025-11-27T23:44:21+00:00'},
 {'Headline': 'Oil in Worst Monthly Run Since 2023 With OPEC+, Ukraine in Focus',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/latest-oil-market-news-and-analysis-for-nov-28',
  'Article': 'Oil headed for the longest run of monthly losses in more than two years, as traders looked ahead to an OPEC+ meeting this weekend and gauged US-led efforts to end the conflict in Ukraine.',
  'Date': '2025-11-27T23:33:36+00:00'},
 {'Headline': 'Vale Boosts Investor Payouts on Strong Output, Iron Ore Prices',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/vale-boosts-investor-payouts-on-strong-output-iron-ore-prices',
  'Article': 'Brazilian iron ore heavyweight Vale SA will pay a special dividend to shareholders on the back of its strong operational performance and high prices of the steelmaking ingredient this year.',
  'Date': '2025-11-27T23:53:06+00:00'},
 {'Headline': 'China EV Profit Woes Fuel Market Anxiety Over Challenging 2026',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/china-ev-profit-woes-fuel-market-anxiety-over-challenging-2026',
  'Article': 'Investors in Chinese electric vehicle stocks had been hoping for a strong earnings season to provide a fresh tailwind. Instead, disappointing results have stoked anxiety about what lies ahead.',
  'Date': '2025-11-27T23:30:00+00:00'},
 {'Headline': 'Emerging Assets Halt Rally in Quiet Session Amid US Holiday',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/china-sets-tone-as-rebound-in-emerging-market-assets-pauses',
  'Article': 'Emerging-market assets halted their recent advance on Thursday, with a gauge of currencies ending the day little changed and stocks falling slightly amid thin liquidity due to the Thanksgiving holiday.',
  'Date': '2025-11-27T11:52:17+00:00'},
 {'Headline': 'How China Came to Dominate Global Shipping Ports',
  'Link': 'https://www.bloomberg.com/news/videos/2025-11-27/china-s-ports-empire-video',
  'Article': 'China has invested billions of dollars in building ports around the globe, giving Beijing a strategic advantage as trade tensions rise, and sparking security concerns for the rest of the world.  (Source: Bloomberg)',
  'Date': '2025-11-27T22:00:03+00:00'},
 {'Headline': 'OBR Says Reeves Had Favorable UK Forecast Before Tax Speech',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/uk-s-obr-says-it-had-no-hand-in-tax-u-turn-that-shocked-markets',
  'Article': 'Rachel Reeves started preparing the public for a manifesto-breaking income-tax rise at the start of November in an unusual pre-budget speech. More surprising was the fact that she did so after privately learning that her fiscal room for maneuver was far better than expected.',
  'Date': '2025-11-27T15:23:41+00:00'},
 {'Headline': 'China Halts Some Brazil Soybean Imports Over Contamination',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/china-halts-some-brazil-soybean-imports-over-contamination',
  'Article': 'China halted soybean imports from five Brazilian plants owned by major global agricultural firms over sanitation concerns, according to people familiar with the matter.',
  'Date': '2025-11-27T14:44:51+00:00'},
 {'Headline': 'Mercuria, Vitol Are Among Bidders for Raizen Argentina Refinery',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/mercuria-vitol-are-among-bidders-for-raizen-argentina-refinery',
  'Article': 'Energy trading giants Mercuria Energy Group and Vitol Group are among finalists for a refinery and hundreds of gas stations in Argentina being sold by Raizen SA, according to people familiar with the matter.',
  'Date': '2025-11-27T19:41:50+00:00'},
 {'Headline': 'Still Constructive on Chinese Equities, Huynh Says',
  'Link': 'https://www.bloomberg.com/news/videos/2025-11-27/still-constructive-on-chinese-equities-huynh-says-video',
  'Article': 'Sophie Huynh, portfolio manager and strategist at BNP Paribas Asset Management, explains why China remains a pocket of diversification in the firm\'s portfolios. She speaks to Lizzy Burden on "Bloomberg Markets." (Source: Bloomberg)',
  'Date': '2025-11-27T19:06:21+00:00'},
 {'Headline': 'Hedge Fund Bond Market Bets Risk Yield Spikes, BIS Chief Warns',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/hedge-fund-bond-market-bets-risk-yield-spikes-bis-chief-warns',
  'Article': 'Bank for International Settlements head Pablo Hernandez de Cos has added his voice to escalating warnings about the role of non-bank firms including hedge funds in sovereign bond markets, at a time of historic government debt levels and a fraught geopolitical backdrop.',
  'Date': '2025-11-27T18:30:00+00:00'},
 {'Headline': 'Anglo-Teck Deal Clears Canada’s National Security Test, Globe Reports',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/anglo-teck-deal-clears-canada-s-national-security-test-globe-reports',
  'Article': 'The Canadian government has cleared Anglo American Plc’s proposed takeover of Teck Resources Ltd. on national security grounds, the Globe and Mail reported, removing one hurdle for the two companies to combine.',
  'Date': '2025-11-27T17:21:22+00:00'},
 {'Headline': 'Credit Market Can Handle Tech’s Debt Deluge, BI Panelists Say',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/credit-market-can-handle-tech-s-debt-deluge-bi-panellists-say',
  'Article': 'Concerns over massive debt issuance from tech giants such as Meta Platforms Inc. and Alphabet Inc. creating oversupply in the credit market are premature, said panelists at the Bloomberg Intelligence European credit market outlook conference in London on Thursday.',
  'Date': '2025-11-27T17:16:31+00:00'},
 {'Headline': 'Lenders Line Up for Chile Copper Smelter in Bet Glut Won’t Last',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/lenders-line-up-for-chile-copper-smelter-in-bet-glut-won-t-last',
  'Article': 'Chile’s Enami said it has received various financing offers for a $1.7 billion copper smelter and will name the banks to structure the deal as early as next week, even as the market remains depressed.',
  'Date': '2025-11-27T16:51:29+00:00'},
 {'Headline': 'Foreign Investment in Canada Falls to Lowest Since Start of ‘24',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/foreign-investment-in-canada-falls-to-lowest-since-start-of-24',
  'Article': 'Foreign direct investment into Canada fell in the third quarter, reaching the lowest level in a year and a half.',
  'Date': '2025-11-27T16:49:38+00:00'},
 {'Headline': 'European Stocks Head for Muted Finish to November; Puma Rallies',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/european-stocks-head-for-muted-finish-to-november-puma-rallies',
  'Article': 'European stocks edged higher on Thursday and were set to close November with a small gain, while Puma SE surged on chatter around a takeover.',
  'Date': '2025-11-27T08:07:35+00:00'},
 {'Headline': 'UK Bonds Fall as Budget Relief Gives Way to Longer-Term Concerns',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/uk-market-declines-offer-reality-check-on-post-budget-relief',
  'Article': 'UK’s bonds fell, ending a five-day rally, as investors analyzed the economic impact of Chancellor Rachel Reeves’s budget that delayed tax-raising measures until later this decade.',
  'Date': '2025-11-27T10:56:28+00:00'},
 {'Headline': 'Moody’s Cuts Brazil’s Raízen to Junk on Rising Debt Levels',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/brazil-s-raizen-cut-to-junk-by-moody-s-on-worsening-debt-metrics',
  'Article': 'Moody’s Ratings downgraded sugar powerhouse Raizen SA, cutting to junk one of Brazil’s last remaining investment-grade corporate credits.',
  'Date': '2025-11-27T13:08:22+00:00'},
 {'Headline': "Bonds Slip\xa0as Investors Assess the UK's Growth Prospects",
  'Link': 'https://www.bloomberg.com/news/live-blog/2025-11-27/uk-budget-2025-reeves-tax-hikes-stocks-pound-steady-markets-today',
  'Article': 'Rachel Reeves presents her budget in Parliament on Nov. 26. Source: UK Parliament',
  'Date': '2025-11-27T06:48:09+00:00'},
 {'Headline': 'Allfunds Said to Draw Takeover Interest From Deutsche Boerse',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/allfunds-said-to-attract-takeover-interest-from-deutsche-boerse',
  'Article': 'European fund distribution platform Allfunds Group Plc is attracting fresh takeover interest from suitors including Deutsche Boerse AG, according to people familiar with the matter.',
  'Date': '2025-11-27T15:18:30+00:00'},
 {'Headline': 'OPEC+ Still on Track to Pause Hikes in Early 2026, Delegates Say',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/opec-still-on-track-to-pause-hikes-in-early-2026-delegates-say',
  'Article': 'OPEC+ nations will probably stick with a decision to pause oil production increases in early 2026 when they meet this weekend, delegates said.',
  'Date': '2025-11-27T15:15:03+00:00'},
 {'Headline': "Westbourne's Whelan on 2026 Market Outlook and Global Stimulus",
  'Link': 'https://www.bloomberg.com/news/videos/2025-11-27/westbourne-s-whelan-on-2026-outlook-and-global-stimulus-video',
  'Article': "Westbourne Research Services Global Macro Strategist, Sharmila Whelan, discusses investment strategies in the context of recent geopolitical and economic developments. She speaks with Bloomberg's Tom Mackenzie on Daybreak Europe. (Source: Bloomberg)",
  'Date': '2025-11-27T14:21:30+00:00'},
 {'Headline': 'ECB Officials Saw Rate Level as Robust Enough to Manage Shocks',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/ecb-officials-saw-rate-level-as-robust-enough-to-manage-shocks',
  'Article': 'European Central Bank officials saw the existing level of borrowing costs as sufficient to deal with possible jolts to the economic outlook, according to an account of their October meeting.',
  'Date': '2025-11-27T12:57:18+00:00'},
 {'Headline': 'South Africa Corn Crop Rises to Second Largest on Record on Rain',
  'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/south-africa-corn-crop-rises-to-second-largest-on-record-on-rain',
  'Article': 'South African farmers probably produced the second-biggest corn crop on record in 2025 as the country experienced good rains earlier in the year.',
  'Date': '2025-11-27T12:56:38+00:00'}]

  return data[:3]

In [12]:
# from processor import NewsProcessor
# from pydantic import BaseModel

# _processor_instance = None

# def get_processor(config: Config) -> NewsProcessor:
#     global _processor_instance
#     if _processor_instance is None:
#         _processor_instance = NewsProcessor(config)
#     return _processor_instance

# class ProcessNewsInput(BaseModel):
#     data: List[BloombergNewsEntry]

# @tool(args_schema=ProcessNewsInput)
# def process_bloomberg_news(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
#     """
#     Run the full NewsProcessor pipeline on raw Bloomberg RSS feed entries.
#     Input: List of dicts with Headline, Link, Article, Date
#     Output: Processed dataframe converted to list[dict]
#     """
#     from config import Config
#     from processor import NewsProcessor

#     config = Config()
#     processor = get_processor(config)

#     # Pipeline
#     # data = processor.remove_redundant_info(data)
#     df = processor.enrich_news_entries_with_classifications(data)
#     df = processor.group_by_date_and_industry(df)
#     df = processor.filter_and_analyze_news(df)
#     df = processor.extract_impactful_news(df, top_n=3)
#     df = processor.get_consolidated_sentiment(df)
#     df = processor.get_explanation(df)

#     return df.to_dict(orient="records")

In [76]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import ToolMessage
import json

from tools.data_feed import get_bloomberg_rss_feeds
from tools.data_processor import process_bloomberg_news
import nest_asyncio
nest_asyncio.apply()

async def run_agent():
  agent = create_react_agent(
      model="gpt-5-nano",
      tools=[get_bloomberg_rss_feeds, get_mock_bloomberg_rss_feeds, process_bloomberg_news]
  )
  instructions  = """1. Fetch MOCK Bloomberg RSS news from the past 1 day.
                    2. Then, process the news.
                    3. Return to me a short summary of pos/neg news impacting the market.
                    """

  last_tool_result = None

  async for chunk in agent.astream({"messages": [{"role": "user", "content": instructions}]}, stream_mode="updates"):
    for step, data in chunk.items():
      print(f"step: {step}")
      print(f"content: {data['messages'][-1].content_blocks}")

      messages = data.get("messages", [])
      if not messages:
          continue

      last_msg = messages[-1]
      if isinstance(last_msg, ToolMessage):
          # Optional: match by tool name
          if last_msg.name == "process_bloomberg_news":
              last_tool_result = last_msg.content
              print("\n=== Extracted tool result ===")
              print(last_tool_result)

  return last_msg.content_blocks[0]["text"], last_tool_result

msg, data = await run_agent()

/tmp/ipython-input-439578570.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


step: agent
content: [{'type': 'tool_call', 'name': 'get_mock_bloomberg_rss_feeds', 'args': {}, 'id': 'call_Fdt4gwtYXZDZjfAy25DMbxJE'}]
step: tools
content: [{'type': 'text', 'text': '[{"Headline": "Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap", "Article": "Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.", "Date": "2025-11-27T22:29:53+00:00"}, {"Headline": "Gold Poised for Fourth Monthly Gain on Fed Rate-Cut Optimism", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/gold-poised-for-fourth-monthly-gain-on-fed-rate-cut-optimism", "Article": "Gold was steady &mdash; and on track for a fourth monthly gain &mdash; as signs the Federal Reserve will cut rates next month supported the precious metal.", "Date": "2025-11-27T23:44:21+00:00"}, {"Headline": "Oil in

Industry Classification: 100%|██████████| 1/1 [00:00<00:00,  9.19batch/s]


Completed processing 3 entries

Dropped 0 (Industry, Date) pairs with Industry='None'
Remaining pairs: 3

Summary Statistics:
Total unique (Industry, Date) pairs: 3
Average articles per pair: 1.00
Max articles in a pair: 1
Min articles in a pair: 1
25th percentile: 1.0
50th percentile: 1.0
75th percentile: 1.0
Number of pairs with at least 3 articles: 0
Total articles: 3


Extracting top 3 impactful news per (Industry, Date) pair...


Processing groups: 100%|██████████| 3/3 [00:00<00:00, 10155.70it/s]


Processing 3 news entries...


FinBERT Sentiment: 100%|██████████| 1/1 [00:00<00:00, 48.87batch/s]


Completed processing 3 entries


Explanation: 100%|██████████| 3/3 [00:16<00:00,  5.39s/it]


step: tools
content: [{'type': 'text', 'text': '[{"Industry": "General Market", "Date": "2025-11-27T22:29:53+00:00", "News": [{"Headline": "Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap", "Date": "2025-11-27T22:29:53+00:00", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap", "Article": "Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.", "SentimentScore": 0.003998962230980396, "Industry": "General Market"}], "ArticleCount": 1, "ImpactfulNews": [{"Headline": "Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap", "Article": "Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.", "SentimentScore": 0.003998962230980396}], "AvgSentimentScore": 0.003998962230980396, "SentimentScore": 0.001944201998412609, "SentimentExplanation": "The s

In [80]:
print(msg)

Here’s a concise summary based on the mock Bloomberg RSS items from the past day and the processed results.

Top-line
- Market tone: Mixed but broadly neutral to slightly cautious. No strong positive catalysts visible in the latest reads.

Key takeaways
- General Market (Asian stocks)
  - Headline: Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap
  - What it suggests: Momentum fading from a recent rally; cautious, range-bound environment.
  - Sentiment signal: Near neutral (about +0.004). Implication: limited directional conviction; await clearer catalysts.

- Commodities and energy
  - Headline: Oil in Worst Monthly Run Since 2023 With OPEC+, Ukraine in Focus
  - What it suggests: Ongoing pressure on oil prices heading into an OPEC+ meeting; geopolitical/energy dynamics remain a headwind.
  - Sentiment signal: Slightly negative (around -0.009). Implication: potential drag on energy equities and energy-linked risk assets; watch for policy decisions at OPEC+.

- Gold and ra

In [78]:
parsed = json.loads(data)
print(f"\nFinal number of entries: {len(parsed)}")


Final number of entries: 3


In [79]:
parsed[2]

{'Industry': 'Financials',
 'Date': '2025-11-27T23:44:21+00:00',
 'News': [{'Headline': 'Gold Poised for Fourth Monthly Gain on Fed Rate-Cut Optimism',
   'Date': '2025-11-27T23:44:21+00:00',
   'Link': 'https://www.bloomberg.com/news/articles/2025-11-27/gold-poised-for-fourth-monthly-gain-on-fed-rate-cut-optimism',
   'Article': 'Gold was steady &mdash; and on track for a fourth monthly gain &mdash; as signs the Federal Reserve will cut rates next month supported the precious metal.',
   'SentimentScore': -0.8583589792251587,
   'Industry': 'Financials'}],
 'ArticleCount': 1,
 'ImpactfulNews': [{'Headline': 'Gold Poised for Fourth Monthly Gain on Fed Rate-Cut Optimism',
   'Article': 'Gold was steady &mdash; and on track for a fourth monthly gain &mdash; as signs the Federal Reserve will cut rates next month supported the precious metal.',
   'SentimentScore': -0.8583589792251587}],
 'AvgSentimentScore': -0.8583589792251587,
 'SentimentScore': -0.867723286151886,
 'SentimentExplanatio